# M2 Notebook 23 — Probabilistic Forecasting

**Status:** Runnable first edition

## Learning objectives

- Produce intervals and quantile forecasts.
- Evaluate coverage, width, pinball loss, and CRPS.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    AutoregressiveModel,bootstrap_forecast_intervals,crps_ensemble,
    interval_width,pinball_loss,prediction_interval_coverage,
)


In [ ]:
rng=np.random.default_rng(23)
t=np.arange(180)
series=30+2*np.sin(2*np.pi*t/12)+rng.normal(scale=2,size=len(t))
train=series[:-24]; test=series[-24:]
model=AutoregressiveModel(12).fit(train)
point=model.forecast(24)
fitted=model.forecast(1)
X,y=__import__("srai_ml").lag_matrix(train,12)
in_sample=model.intercept_+X@model.coef_
residuals=y-in_sample
lower,upper=bootstrap_forecast_intervals(point,residuals,24,3000,.9,seed=23)
{"coverage":prediction_interval_coverage(test,lower,upper),
 "mean_width":interval_width(lower,upper)}


In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
x=np.arange(len(train),len(series))
ax.plot(x,test,label="actual")
ax.plot(x,point,label="point forecast")
ax.fill_between(x,lower,upper,alpha=.3,label="90% interval")
ax.legend(); ax.set_title("Probabilistic Forecast")
plt.show()


## Quantile loss

In [ ]:
{"lower_pinball":pinball_loss(test,lower,.05),
 "upper_pinball":pinball_loss(test,upper,.95)}


## Decision Intelligence case

Planning should use forecast distributions or intervals when under- and over-provisioning have asymmetric costs.

## Key insight

Probabilistic forecasts communicate uncertainty directly and should be evaluated for both calibration and sharpness.